# Bangalore Housing Prices Analysis

## Project Objective
Analyze Bangalore housing price data using Python to understand data quality, clean important columns, treat outliers, and explore relationships between property features and price.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

## 2. Load Dataset

Keep `BHP.csv` in the same folder as this notebook before running the project.

In [ ]:
BHP = pd.read_csv("BHP.csv")
BHP.head()

## 3. Data Understanding

In [ ]:
print("Rows and Columns:", BHP.shape)

In [ ]:
BHP.info()

In [ ]:
BHP.dtypes

In [ ]:
BHP.isnull().sum()

In [ ]:
BHP.duplicated().sum()

### Initial Observations
- The dataset contains both numerical and categorical columns.
- `total_sqft` needs cleaning because it contains ranges and unit-based values.
- Some columns contain missing values and need treatment.
- Duplicate rows should be checked and removed before analysis.

## 4. Data Cleaning - Total Square Feet Conversion

In [ ]:
BHP['total_sqft'].unique()[:20]

In [ ]:
def convert_sqft(value):
    """Convert total_sqft values into numeric square feet."""
    try:
        value = str(value).strip()
        
        # Case 1: Range values like "1133 - 1384"
        if '-' in value:
            nums = value.split('-')
            return (float(nums[0]) + float(nums[1])) / 2
        
        # Case 2: Unit-based values
        conversion_factors = {
            "Acres": 43560,
            "Sq. Meter": 10.7639,
            "Sq. Yards": 9,
            "Perch": 272.25,
            "Cents": 435.6,
            "Guntha": 1089,
            "Grounds": 2400
        }
        
        match = re.match(r"([\d\.]+)\s*([A-Za-z. ]+)", value)
        if match:
            num, unit = match.groups()
            unit = unit.strip()
            return float(num) * conversion_factors.get(unit, np.nan)
        
        # Case 3: Plain number
        return float(value)
    except:
        return np.nan

BHP['converted_sqft'] = BHP['total_sqft'].apply(convert_sqft)
BHP[['total_sqft', 'converted_sqft']].head()

## 5. Missing Value Treatment

In [ ]:
BHP.isnull().sum()

In [ ]:
# Fill balcony missing values with 0 because some properties may not have balconies
BHP['balcony'] = BHP['balcony'].fillna(0)

# Drop rows where important fields are missing
BHP = BHP.dropna(subset=['location', 'size', 'bath', 'price', 'converted_sqft'])

BHP.isnull().sum()

## 6. Feature Engineering

In [ ]:
# Extract BHK from size column
BHP['bhk'] = BHP['size'].apply(lambda x: int(str(x).split()[0]))

# Calculate price per square foot
BHP['price_per_sqft'] = (BHP['price'] * 100000) / BHP['converted_sqft']

BHP[['size', 'bhk', 'price', 'converted_sqft', 'price_per_sqft']].head()

## 7. Duplicate Treatment

In [ ]:
print("Duplicate rows before removal:", BHP.duplicated().sum())
BHP.drop_duplicates(inplace=True, keep='first')
print("Duplicate rows after removal:", BHP.duplicated().sum())

## 8. Statistical Summary

In [ ]:
BHP[['price', 'converted_sqft', 'bath', 'balcony', 'bhk', 'price_per_sqft']].describe()

In [ ]:
print("Mean Price:", BHP['price'].mean())
print("Median Price:", BHP['price'].median())
print("Mode Price:", BHP['price'].mode()[0])

## 9. Univariate Analysis

In [ ]:
BHP['area_type'].value_counts().plot(kind='bar', figsize=(8,5))
plt.title("Area Type Distribution")
plt.xlabel("Area Type")
plt.ylabel("Count")
plt.show()

In [ ]:
BHP['bhk'].value_counts().sort_index().plot(kind='bar', figsize=(8,5))
plt.title("BHK Distribution")
plt.xlabel("BHK")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(BHP['price'], kde=True)
plt.title("Price Distribution")
plt.xlabel("Price in Lakhs")
plt.show()

## 10. Outlier Detection and Treatment

In [ ]:
plt.figure(figsize=(6,5))
plt.boxplot(BHP['price'])
plt.title("Price Boxplot Before Outlier Treatment")
plt.ylabel("Price")
plt.show()

In [ ]:
Q1 = BHP['price'].quantile(0.25)
Q3 = BHP['price'].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

In [ ]:
new_BHP = BHP[(BHP['price'] >= lower_limit) & (BHP['price'] <= upper_limit)]
print("Shape before outlier treatment:", BHP.shape)
print("Shape after outlier treatment:", new_BHP.shape)

In [ ]:
plt.figure(figsize=(6,5))
plt.boxplot(new_BHP['price'])
plt.title("Price Boxplot After Outlier Treatment")
plt.ylabel("Price")
plt.show()

## 11. Bivariate Analysis

In [ ]:
new_BHP.groupby('area_type')['price'].mean().sort_values(ascending=False).plot(kind='bar', figsize=(8,5))
plt.title("Average Price by Area Type")
plt.xlabel("Area Type")
plt.ylabel("Average Price")
plt.show()

In [ ]:
new_BHP.groupby('bhk')['price'].mean().plot(kind='bar', figsize=(8,5))
plt.title("Average Price by BHK")
plt.xlabel("BHK")
plt.ylabel("Average Price")
plt.show()

In [ ]:
new_BHP.groupby('location')['price'].mean().sort_values(ascending=False).head(10).plot(kind='bar', figsize=(10,5))
plt.title("Top 10 Locations by Average Price")
plt.xlabel("Location")
plt.ylabel("Average Price")
plt.xticks(rotation=45, ha='right')
plt.show()

## 12. Skewness, Variance and Standard Deviation

In [ ]:
print("Variance:", new_BHP['price'].var())
print("Standard Deviation:", new_BHP['price'].std())
print("Skewness before outlier treatment:", BHP['price'].skew())
print("Skewness after outlier treatment:", new_BHP['price'].skew())
print("Kurtosis:", new_BHP['price'].kurtosis())

## 13. Correlation Analysis

In [ ]:
num_cols = new_BHP.select_dtypes(include=np.number).columns
new_BHP[num_cols].corr()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(new_BHP[num_cols].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

## 14. Conclusion

- The dataset required cleaning because `total_sqft` contained ranges and different measurement units.
- Missing values and duplicate records were treated before analysis.
- Most properties are concentrated around common BHK types such as 2 BHK and 3 BHK.
- Property prices vary significantly across area types and locations.
- Outlier treatment helped reduce extreme price influence.
- Correlation analysis helps understand how numerical features such as square feet, BHK, bathrooms, and price are related.